In [721]:
import csv
import random
import math

In [722]:
def layer_pass(input_layer, weights):
    output_layer_size = len(weights)
    output_layer_sums = [0 for i in range(output_layer_size)]
    for o in range(output_layer_size):
        for i in range(len(input_layer)):
            output_layer_sums[o] += weights[o][i] * input_layer[i]
    return output_layer_sums

In [723]:
def apply_activation_function(sums, mutator):
    for s in range(len(sums)):
        sums[s] = mutator(sums[s])
    return sums
def relu(val):
    if val < 0:
        return 0
    return val
def relu_p(val):
    if val > 0:
        return 1
    return 0
def sigmoid(val):
    return 1 / (1 + math.exp(-1 * val))
def sigmoid_p(val):
    s = val
    return s * (1 - s)
def tanh(val):
    return math.tanh(val)
def tanh_p(val):
    return 1 - math.pow(val, 2)

In [666]:
def create_network_weights(layer_sizes, modifier = 0.1):
    weights = [[[((random.random() * 2 - 1) * modifier) for i in range(layer_sizes[l-1])] for c in range(layer_sizes[l])] for l in range(1, len(layer_sizes))]
    return weights

In [667]:
def pass_values(input_values, network_weights, functions):
    values = input_values
    activation_values = []
    for l in range(len(network_weights)):
        values = layer_pass(values, network_weights[l])
        if l != len(network_weights) - 1:
            values = apply_activation_function(values, functions[l])
        activation_values.append(values)
    return activation_values

In [668]:
def softmax(values):
    max_value = max(values)
    values = [math.exp(x - max_value) for x in values]
    values_sum = sum(values)
    values = [x / values_sum for x in values]
    return values

In [669]:
def cross_entropy_loss(prediction, actual):
    return -1 * sum([actual[x] * math.log(prediction[x]) for x in range(len(actual))])

In [670]:
def error_gradient(prediction, actual):
    return [prediction[x] - actual[x] for x in range(len(actual))]

In [671]:
def back_propogate_layer(error, activation_values, weights, learning_rate, activation_function_derivative):
    gradient_matrix = output_gradients(error, activation_values)
    updated_weights = update_weights(weights, learning_rate, gradient_matrix)
    
    internal_layer_error = layer_error(weights, error)
    new_error = apply_activation_function(internal_layer_error, activation_function_derivative)
    
    return new_error, updated_weights

In [672]:
def back_propogate_network(input_values, network_weights, functions, functions_p, output_values):
    activation_values = pass_values(input_values, network_weights, functions)
    network_output = activation_values[-1]
    softmaxed_values = softmax(network_output)
    loss = cross_entropy_loss(softmaxed_values, output_values)
    error = error_gradient(softmaxed_values, output_values)
    error_i = error

    new_weights = [0 for l in range(len(network_weights))]

    for i in range(-1, -1-len(network_weights), -1):
        layer_weights = network_weights[i]
        
        activation_sums = []
        if i != (-1 * len(network_weights)):
            activation_sums = activation_values[i-1]
        else:
             activation_sums = input_values
    
        error, new_layer_weights = back_propogate_layer(error, activation_sums, layer_weights, 0.01, sigmoid_p)
        new_weights[i] = new_layer_weights

    return new_weights, loss, error_i
    

In [673]:
def output_gradients(error, activation_values):
    return[[e * v for v in activation_values] for e in error]

In [674]:
def update_weights(weights, learning_rate, gradient_matrix):
    return [[weights[n][c] - (gradient_matrix[n][c] * learning_rate) for c in range(len(weights[n]))] for n in range(len(weights))]

In [675]:
def layer_error(weights, error):
    return [sum([error[i] * weights[i][e] for i in range(len(weights))]) for e in range(len(weights[0]))]

# Begin network

In [724]:
type_map = {
    "setosa": 0,
    "versicolor": 1,
    "virginica": 2
}
data = []

with open('iris.csv', newline='') as csvfile:
    spamreader = csv.reader(csvfile, delimiter=' ', quotechar='|')
    for row in spamreader:
        split_data = (row[0].split(","))
        try:
            output_row = [float(split_data[0]),float(split_data[1]),float(split_data[2]),float(split_data[3]), type_map[split_data[4]]]
        except:
            pass
        data.append(output_row)

data_points = len(data)

print("Data header")
data[:5]

Data header


[[5.9, 3.0, 5.1, 1.8, 2],
 [5.1, 3.5, 1.4, 0.2, 0],
 [4.9, 3.0, 1.4, 0.2, 0],
 [4.7, 3.2, 1.3, 0.2, 0],
 [4.6, 3.1, 1.5, 0.2, 0]]

In [726]:
max_value = 0
min_value = float('inf')
for i in range(0, len(data)):
    if max(data[i][:4]) > max_value:
        max_value = max(data[i][:4])
    if min(data[i][:4]) < min_value:
        min_value = min(data[i][:4])

delta = max_value - min_value
data = [[ [((data[d][v] - min_value) / delta), data[d][v]][v==4] for v in range(len(data[d]))] for d in range(len(data))]

print("Data header")
data[:5]

Data header


[[0.7435897435897436,
  0.37179487179487175,
  0.641025641025641,
  0.21794871794871792,
  2],
 [0.641025641025641,
  0.43589743589743585,
  0.16666666666666663,
  0.01282051282051282,
  0],
 [0.6153846153846154,
  0.37179487179487175,
  0.16666666666666663,
  0.01282051282051282,
  0],
 [0.5897435897435898,
  0.3974358974358974,
  0.15384615384615383,
  0.01282051282051282,
  0],
 [0.5769230769230769,
  0.3846153846153846,
  0.17948717948717946,
  0.01282051282051282,
  0]]

In [727]:
random.shuffle(data)
input_data, output_data = [], []
for i in range(data_points):
    input_data.append(data[i][:4])
    output_data.append([[0, 1][t == data[i][4]] for t in range(3)])

In [728]:
train_x, train_y = input_data[:math.floor(data_points * 0.7)], output_data[:math.floor(data_points * 0.7)]
test_x, test_y = input_data[math.floor(data_points * 0.7):], output_data[math.floor(data_points * 0.7):]

In [765]:
network_weights = create_network_weights([4,5,3], 0.5)

functions = [relu]
functions_p = [relu_p]
learning_rate = 0.001

In [767]:
sets = 100
loss = 0

total_backpropogations = sets * len(train_x)
backpropogations_complete = 0

for s in range(0, sets):
    for r in range(len(train_x)):
        network_weights, loss, error = back_propogate_network(train_x[r], network_weights, functions, functions_p, train_y[r])
        backpropogations_complete += 1

        if backpropogations_complete % 10 == 0:
            print(loss)

0.8064585537393035
0.7730827436850106
1.2537877513019726
0.7400272268151677
1.1374703180734458
1.2531890132078918
1.2456471404517522
1.0800880373192179
1.004375987046559
1.0389530149896504
0.9681407687468301
0.7466218409667778
1.262638598281765
1.237937225555359
1.0744060914431197
1.1162526282437257
1.0976976634769482
0.6704747968749337
1.002124632510763
0.7839024318736376
0.7804987411245783
0.8092760724799282
0.7641608325955502
1.21600655350814
0.7424357754070767
1.1420227947907111
1.1841484818037915
1.1823635978218914
1.0720319898403803
0.9734123813669022
1.0207918666363778
0.9383151023924595
0.7458242680616235
1.1968203844594314
1.172661114752774
1.0731477099905795
1.1207995247368607
1.0989757558851005
0.6521726581902998
0.9758649670927785
0.7957562852861767
0.7929157726858803
0.8152302894546016
0.7539805039645823
1.1504874653006847
0.7478013106430252
1.1631653982667784
1.0845228777819669
1.0812118900765377
1.070184240049846
0.9428410413777392
1.0078697992882482
0.9092580083272088
0

ValueError: math domain error

In [768]:
softmax(pass_values(test_x[1], network_weights, functions)[-1])

[1.6531950529786502e-254, 3.774673020691166e-216, 1.0]

In [769]:
test_y[1]

[1, 0, 0]

In [719]:
total_backpropogations

105000

In [720]:
network_weights

[[[-192.57847102259615,
   -97.53094771451039,
   -124.90701952549858,
   -37.684282661858795],
  [-192.3009247589908,
   -97.42590176695532,
   -124.75863492549279,
   -37.92735435079476],
  [-191.09656997639468,
   -96.83813187586249,
   -123.75522254261804,
   -37.63057996971744],
  [-192.20006606259136,
   -97.48931553813868,
   -124.72654024489688,
   -37.70173659213542],
  [-191.35666364707149,
   -96.7643027588553,
   -124.3373913977411,
   -37.705907317504824]],
 [[-0.11088086227413699,
   -0.07236376708136595,
   -0.1645526147621343,
   0.09349526582475609,
   0.16723129506816298],
  [-0.11179440002362884,
   -0.011248060673942378,
   -0.1314339938783757,
   -0.014241271246379033,
   -0.18240865249824822],
  [-0.14833587181587435,
   -0.060100902040982764,
   0.19092521494317097,
   -0.06748221969911065,
   -0.18968555547187746]]]